In [1]:
# 1. Di chuyển lại vào thư mục mã nguồn NVIDIA mà bạn vừa clone
%cd /tmp/TensorRT/tools/pytorch-quantization

# 2. Cài đặt phiên bản setuptools cũ (< 70) để khôi phục module distutils cho Python 3.12
!pip install "setuptools<70.0.0" wheel

# 3. Biên dịch mã nguồn KHÔNG sử dụng môi trường cách ly (bắt buộc dùng setuptools vừa cài)
!pip install --no-build-isolation .

# 4. Di chuyển an toàn trở lại thư mục dự án của bạn
%cd /content/drive/MyDrive/ppe_cv/ppe_v2

/tmp/TensorRT/tools/pytorch-quantization
Processing ./.
  Preparing metadata (pyproject.toml) ... done
  Using cached sphinx_glpi_theme-0.6-py2.py3-none-any.whl.metadata (1.8 kB)
Using cached sphinx_glpi_theme-0.6-py2.py3-none-any.whl (4.2 MB)
  Created wheel for pytorch_quantization: filename=pytorch_quantization-2.2.1-cp312-cp312-linux_x86_64.whl size=3356725 sha256=5e6c4803eb2257ad437c2caabd240a485ed1179b92cdcb79e0edf1aa5700fcb4
  Stored in directory: /tmp/pip-ephem-wheel-cache-g_oe5lll/wheels/7f/72/81/7f89b6b52f48942737a1d0f7a2eaa861289f521ece19a112d0
Successfully built pytorch_quantization
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pytorch_quantization]
/content/drive/MyDrive/ppe_cv/ppe_v2


In [2]:
# 1. Check GPU allocation
!nvidia-smi

# 2. Mount Google Drive to persist data and access the repository
from google.colab import drive
drive.mount('/content/drive')

Mon Aug  3 03:34:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   25C    P0             45W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# 1. Di chuyển vào thư mục chứa custom Ultralytics đã upload
# Hãy thay thế đường dẫn bên dưới bằng vị trí thực tế của bạn
%cd /content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices
# 2. Gỡ bản gốc để tránh xung đột
!pip uninstall -y ultralytics

# 3. Cài đặt bản custom (lệnh này sẽ cài đặt thư viện vào hệ thống Colab)
!pip install -e .

/content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices
Found existing installation: ultralytics 8.3.231
Uninstalling ultralytics-8.3.231:
  Successfully uninstalled ultralytics-8.3.231
Obtaining file:///content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.231-0.editable-py3-none-any.whl size=11709 sha256=2149fc9d69c2ada8586f77b2783cc0e0a63e705fe6d54a3675282043c6fc5bf8
  Stored in directory: /tmp/pip-ephem-wheel-cache-ubaurhsp/wheels/6d/54/88/b24c9a513438bdb6003f090cbfc70bfc9b5fc534848fa36efd
Successfully built ultralytics


In [4]:
# Di chuyển về thư mục gốc của dự án
%cd /content/drive/MyDrive/ppe_cv/ppe_v2

# Kiểm tra lại xem đã đứng đúng thư mục chưa (sẽ thấy thư mục scripts, file data.yaml...)
!ls

/content/drive/MyDrive/ppe_cv/ppe_v2
assets	     data.yaml			requirements.txt  yolo11n.pt
bench.ipynb  demo.py			runs		  yolov5_au.ipynb
config.py    experiments_dagshub.ipynb	scripts		  yolov8_baseline.ipynb
configs      README.md			weights		  yolov8_prune.ipynb


In [5]:
!pip install mlflow

In [7]:
import os
import sys

# 1. Khai báo đường dẫn tới thư mục repo custom Ultralytics
custom_repo_path = "/content/drive/MyDrive/ppe_cv/Pruned-QAT-for-YOLOv8-on-waste-detection-tasks-for-edge-devices"
if custom_repo_path not in sys.path:
    sys.path.insert(0, custom_repo_path)

# 2. Di chuyển vào thư mục dự án ppe_v2
%cd /content/drive/MyDrive/ppe_cv/ppe_v2

# 3. Cho phép MLflow File Store
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

/content/drive/MyDrive/ppe_cv/ppe_v2


In [8]:
# Thực thi huấn luyện QAT trên Colab
!python scripts/qat.py \
    --pruned-checkpoint /content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.30.pt \
    --pretrained-weight /content/drive/MyDrive/ppe_cv/ppe_v2/runs/train-finetune/weights/best.pt \
    --data-config data.yaml \
    --epochs 10 \
    --batch 32 \
    --workers 4 \
    --device 0

[QAT-Pruned] Pruned checkpoint: /content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.30.pt
[QAT-Pruned] Finetuned weight: /content/drive/MyDrive/ppe_cv/ppe_v2/runs/train-finetune/weights/best.pt
Ultralytics 8.3.231 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=0.04, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=2.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ppe_cv/ppe_v2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.2, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, finetune=None, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.1, 

In [10]:
!pip install onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 151.4 MB/s  0:00:00


In [11]:
# Xuất mô hình QAT sang file ONNX
!python scripts/export_onnx.py \
    --pruned-checkpoint /content/drive/MyDrive/ppe_cv/ppe_v2/weights/pruned_r0.30.pt \
    --weight /content/drive/MyDrive/ppe_cv/ppe_v2/runs/qat-pruned/train/weights/best.pt


                   from  n    params  module                                            arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv                  [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv                  [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block_pruned.C2fPruned     [96, 96, [48, 48], [48, 48], [48, 48], 96, 2, True]
  3                  -1  1    162808  ultralytics.nn.modules.conv.Conv                  [96, 188, 3, 2]               
  4                  -1  4    810814  ultralytics.nn.modules.block_pruned.C2fPruned     [188, 192, [96, 96], [95, 96, 96, 96], [96, 96, 96, 96], 192, 4, True]
  5                  -1  1    629720  ultralytics.nn.modules.conv.Conv                  [192, 364, 3, 2]              
  6                  -1  4   3235194  ultralytics.nn.modules.block_pruned.C2fPruned     [364, 384, [192, 

In [12]:
import mlflow
import os
import getpass

DAGSHUB_USER = input("Nhập DagsHub Username: ")
DAGSHUB_TOKEN = getpass.getpass("Nhập DagsHub Token: ")
DAGSHUB_REPO = input("Nhập tên DagsHub Repo (VD: ppe-project): ")

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow"

Nhập DagsHub Username: TQanh23
Nhập DagsHub Token: ··········
Nhập tên DagsHub Repo (VD: ppe-project): ppe_v2


In [14]:
mlflow.set_experiment("YOLOv8_Quantization")

with mlflow.start_run(run_name="Phase4_QAT_Export"):
    # Ghi nhận các tham số cấu hình cơ bản của bước export
    mlflow.log_params({
        "mode": "qat_export",
        "format": "onnx_tensorrt",
        "int8": True
    })

    # Đường dẫn tới thư mục QAT và file ONNX vừa tạo
    qat_run_dir = "/content/drive/MyDrive/ppe_cv/ppe_v2/runs/qat-pruned/train"
    qat_onnx_file = os.path.join(qat_run_dir, "weights/best.tensorrt.onnx")

    # 1. Upload riêng file ONNX để dễ tìm
    if os.path.exists(qat_onnx_file):
        mlflow.log_artifact(qat_onnx_file, artifact_path="qat_exported_models")
        print(f"✅ Đã log thành công file ONNX: {qat_onnx_file}")
    else:
        print(f"❌ Không tìm thấy file ONNX tại {qat_onnx_file}. Hãy kiểm tra lại đường dẫn.")

    # 2. Upload toàn bộ log huấn luyện, biểu đồ, metrics của quá trình QAT (nếu có)
    if os.path.exists(qat_run_dir):
        mlflow.log_artifacts(local_dir=qat_run_dir, artifact_path="qat_training_results")
        print("✅ Đã log toàn bộ thư mục kết quả huấn luyện QAT lên MLflow!")

✅ Đã log thành công file ONNX: /content/drive/MyDrive/ppe_cv/ppe_v2/runs/qat-pruned/train/weights/best.tensorrt.onnx
✅ Đã log toàn bộ thư mục kết quả huấn luyện QAT lên MLflow!
🏃 View run Phase4_QAT_Export at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/11/runs/32b35f5293594871af24ca5c1c340308
🧪 View experiment at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/11
